# 03 - MS-ASL Full Download (RAW Videos) and Universal CSV

This notebook downloads RAW MS-ASL videos once per unique source video and maps them to all annotation rows.

Configured decisions:
- full dataset (train + val + test),
- no clip cutting in this stage,
- one downloaded video can serve many rows,
- download only if the video is not already present locally,
- output directory for RAW videos is separate (`videos_MS_ASL_raw`),
- 8 workers,
- no cache for yt-dlp,
- output includes all rows, including has_video = False,
- final CSV schema matches universal metadata exactly.

In [1]:
from __future__ import annotations

from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from urllib.parse import parse_qs, urlparse
import hashlib
import json
import shutil
import subprocess
import tempfile
from typing import Any

import cv2
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)

In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent.parent if Path.cwd().name == 'MS-ASL_EDA' else (Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve())
MSASL_DIR = PROJECT_ROOT / 'MS-ASL'
TRAIN_JSON = MSASL_DIR / 'MSASL_train.json'
VAL_JSON = MSASL_DIR / 'MSASL_val.json'
TEST_JSON = MSASL_DIR / 'MSASL_test.json'

OUTPUT_VIDEO_DIR = MSASL_DIR / 'videos_MS_ASL_raw'
OUTPUT_MANIFEST_CSV = MSASL_DIR / 'msasl_download_manifest_full_raw_rows.csv'
OUTPUT_UNIQUE_VIDEO_MANIFEST_CSV = MSASL_DIR / 'msasl_download_manifest_full_raw_unique_videos.csv'
OUTPUT_UNIVERSAL_CSV = MSASL_DIR / 'msasl_universal_metadata_full_all_raw.csv'
OUTPUT_UNIVERSAL_CSV_HAS_VIDEO = MSASL_DIR / 'msasl_universal_metadata_full_all_raw_has_video.csv'

MAX_WORKERS = 8
OVERWRITE = False
SOURCE_NAME = 'msasl'
LABEL_COLUMN = 'clean_text'
YTDLP_QUALITY = 'best[height<=720]/best[height<=480]/best'
YTDLP_NO_CACHE = True

print('PROJECT_ROOT:', PROJECT_ROOT)
print('MSASL_DIR exists:', MSASL_DIR.exists())
print('OUTPUT_VIDEO_DIR:', OUTPUT_VIDEO_DIR)
print('OUTPUT_MANIFEST_CSV:', OUTPUT_MANIFEST_CSV)
print('OUTPUT_UNIQUE_VIDEO_MANIFEST_CSV:', OUTPUT_UNIQUE_VIDEO_MANIFEST_CSV)
print('OUTPUT_UNIVERSAL_CSV:', OUTPUT_UNIVERSAL_CSV)
print('MAX_WORKERS:', MAX_WORKERS, '| OVERWRITE:', OVERWRITE)

PROJECT_ROOT: C:\Users\Magda\source\repos\private\szum
MSASL_DIR exists: True
OUTPUT_VIDEO_DIR: C:\Users\Magda\source\repos\private\szum\MS-ASL\videos_MS_ASL_raw
OUTPUT_MANIFEST_CSV: C:\Users\Magda\source\repos\private\szum\MS-ASL\msasl_download_manifest_full_raw_rows.csv
OUTPUT_UNIQUE_VIDEO_MANIFEST_CSV: C:\Users\Magda\source\repos\private\szum\MS-ASL\msasl_download_manifest_full_raw_unique_videos.csv
OUTPUT_UNIVERSAL_CSV: C:\Users\Magda\source\repos\private\szum\MS-ASL\msasl_universal_metadata_full_all_raw.csv
MAX_WORKERS: 8 | OVERWRITE: False


In [3]:
def check_required_tools() -> None:
    missing = []
    if shutil.which('yt-dlp') is None:
        missing.append('yt-dlp')
    if shutil.which('ffmpeg') is None:
        missing.append('ffmpeg')

    if missing:
        raise EnvironmentError(f'Missing required tools in PATH: {missing}')

    print('Tools OK: yt-dlp, ffmpeg')


def load_split(path: Path, split_name: str) -> pd.DataFrame:
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    df['split'] = split_name
    return df


def normalize_url(raw_url: Any) -> str:
    url = str(raw_url).strip()
    if not url:
        return ''
    low = url.lower()
    if low.startswith('http://') or low.startswith('https://'):
        return url
    if low.startswith('www.'):
        return 'https://' + url
    if 'youtube.com/' in low or 'youtu.be/' in low:
        return 'https://' + url
    return url


def extract_video_id(url: str) -> str:
    if not url:
        return ''
    try:
        parsed = urlparse(url)
        host = parsed.netloc.lower()
        path = parsed.path.strip('/')

        if 'youtu.be' in host and path:
            return path.split('/')[0]

        if 'youtube.com' in host:
            qs = parse_qs(parsed.query)
            if 'v' in qs and qs['v']:
                return qs['v'][0]
            if path.startswith('shorts/'):
                return path.split('/')[1] if len(path.split('/')) > 1 else ''
            if path.startswith('embed/'):
                return path.split('/')[1] if len(path.split('/')) > 1 else ''
    except Exception:
        return ''
    return ''


def make_video_key(url_norm: str) -> str:
    video_id = extract_video_id(url_norm)
    if video_id:
        return video_id
    return hashlib.sha1(url_norm.encode('utf-8')).hexdigest()[:16]


def yt_dlp_download(url: str, temp_dir: Path) -> Path:
    output_template = str(temp_dir / 'raw.%(ext)s')
    cmd = [
        'yt-dlp',
        '--no-playlist',
        '-f', YTDLP_QUALITY,
        '-o', output_template,
    ]
    if YTDLP_NO_CACHE:
        cmd.append('--no-cache-dir')
    cmd.append(url)

    subprocess.run(cmd, check=True, capture_output=True, text=True)

    candidates = sorted(temp_dir.glob('raw.*'))
    if not candidates:
        raise FileNotFoundError('yt-dlp finished but no output file found')
    return candidates[0]


def find_existing_video(video_key: str) -> Path | None:
    matches = sorted(OUTPUT_VIDEO_DIR.glob(f'msasl_raw_{video_key}.*'))
    if matches:
        return matches[0]
    return None


def probe_video(video_path: Path) -> tuple[float | None, int | None, int | None, int | None]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None, None, None, None

    fps = float(cap.get(cv2.CAP_PROP_FPS))
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    if fps <= 0:
        fps = None
    if frames <= 0:
        frames = None
    if width <= 0:
        width = None
    if height <= 0:
        height = None

    return fps, frames, width, height


def to_repo_path(absolute_path: str, project_root: Path) -> str:
    """Returns a path relative to project_root, using forward slashes."""
    if not absolute_path:
        return ''
    try:
        return str(Path(absolute_path).relative_to(project_root)).replace('\\', '/')
    except ValueError:
        return str(absolute_path).replace('\\', '/')


def process_video(video_key: str, url: str) -> dict[str, Any]:
    result = {
        'video_key': video_key,
        'url_norm': url,
        'local_video_path': '',
        'has_video': False,
        'download_status': 'pending',
        'attempts_used': 0,
        'error_message': '',
        'fps_out': np.nan,
        'frame_count_out': np.nan,
        'video_width_out': np.nan,
        'video_height_out': np.nan,
        'duration_sec_out': np.nan,
    }

    if not url:
        result['download_status'] = 'invalid_url'
        result['error_message'] = 'Empty URL'
        return result

    existing = find_existing_video(video_key)
    if existing is not None and existing.exists() and not OVERWRITE:
        fps, frames, width, height = probe_video(existing)
        result['local_video_path'] = to_repo_path(str(existing), PROJECT_ROOT)
        result['download_status'] = 'already_exists'
        result['has_video'] = fps is not None and frames is not None
        result['attempts_used'] = 0
        result['fps_out'] = fps
        result['frame_count_out'] = frames
        result['video_width_out'] = width
        result['video_height_out'] = height
        if fps and frames:
            result['duration_sec_out'] = frames / fps
        return result

    try:
        result['attempts_used'] = 1
        with tempfile.TemporaryDirectory(prefix='msasl_raw_') as tmp:
            tmp_dir = Path(tmp)
            raw_video = yt_dlp_download(url, tmp_dir)
            suffix = raw_video.suffix.lower() if raw_video.suffix else '.mp4'
            output_path = OUTPUT_VIDEO_DIR / f'msasl_raw_{video_key}{suffix}'
            shutil.move(str(raw_video), str(output_path))

        fps, frames, width, height = probe_video(output_path)
        result['local_video_path'] = to_repo_path(str(output_path), PROJECT_ROOT)
        result['has_video'] = fps is not None and frames is not None
        result['download_status'] = 'ok' if result['has_video'] else 'probe_failed'
        result['fps_out'] = fps
        result['frame_count_out'] = frames
        result['video_width_out'] = width
        result['video_height_out'] = height
        if fps and frames:
            result['duration_sec_out'] = frames / fps
        if not result['has_video'] and not result['error_message']:
            result['error_message'] = 'Downloaded file could not be probed by OpenCV'

    except Exception as exc:
        result['download_status'] = 'failed'
        result['error_message'] = str(exc)[:1200]

    return result

In [4]:
check_required_tools()

df_train = load_split(TRAIN_JSON, 'train')
df_val = load_split(VAL_JSON, 'val')
df_test = load_split(TEST_JSON, 'test')

df = pd.concat([df_train, df_val, df_test], ignore_index=True)
df['clean_text'] = df['clean_text'].astype(str).str.strip().str.lower()
df['url_norm'] = df['url'].apply(normalize_url)
df['video_id'] = df['url_norm'].apply(extract_video_id)
df['video_key'] = df['url_norm'].apply(make_video_key)

for c in ['start', 'end', 'signer_id', 'fps', 'width', 'height', 'start_time', 'end_time']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

OUTPUT_VIDEO_DIR.mkdir(parents=True, exist_ok=True)

video_jobs = df[['video_key', 'url_norm']].drop_duplicates().reset_index(drop=True)
print('Total annotation rows:', len(df))
print('Unique videos to process:', len(video_jobs))

video_records: list[dict[str, Any]] = []
total_videos = len(video_jobs)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = [
        ex.submit(process_video, row['video_key'], row['url_norm'])
        for _, row in video_jobs.iterrows()
    ]

    done = 0
    for fut in as_completed(futures):
        video_records.append(fut.result())
        done += 1
        if done % 100 == 0 or done == total_videos:
            print(f'Processed videos {done}/{total_videos}')

video_manifest_df = pd.DataFrame(video_records)
video_manifest_df.to_csv(OUTPUT_UNIQUE_VIDEO_MANIFEST_CSV, index=False)

manifest_df = df.merge(video_manifest_df, on=['video_key', 'url_norm'], how='left')
manifest_df['row_index'] = np.arange(len(manifest_df))
manifest_df = manifest_df.sort_values('row_index').reset_index(drop=True)
manifest_df.to_csv(OUTPUT_MANIFEST_CSV, index=False)

print('Saved unique-video manifest:', OUTPUT_UNIQUE_VIDEO_MANIFEST_CSV)
print('Saved row-level manifest:', OUTPUT_MANIFEST_CSV)
display(manifest_df.head(10))

Tools OK: yt-dlp, ffmpeg
Total annotation rows: 25513
Unique videos to process: 7213
Processed videos 100/7213
Processed videos 200/7213
Processed videos 300/7213
Processed videos 400/7213
Processed videos 500/7213
Processed videos 600/7213
Processed videos 700/7213
Processed videos 800/7213
Processed videos 900/7213
Processed videos 1000/7213
Processed videos 1100/7213
Processed videos 1200/7213
Processed videos 1300/7213
Processed videos 1400/7213
Processed videos 1500/7213
Processed videos 1600/7213
Processed videos 1700/7213
Processed videos 1800/7213
Processed videos 1900/7213
Processed videos 2000/7213
Processed videos 2100/7213
Processed videos 2200/7213
Processed videos 2300/7213
Processed videos 2400/7213
Processed videos 2500/7213
Processed videos 2600/7213
Processed videos 2700/7213
Processed videos 2800/7213
Processed videos 2900/7213
Processed videos 3000/7213
Processed videos 3100/7213
Processed videos 3200/7213
Processed videos 3300/7213
Processed videos 3400/7213
Proces

,org_text,clean_text,start_time,signer_id,signer,start,end,file,label,height,fps,end_time,url,text,box,width,review,split,url_norm,video_id,video_key,local_video_path,has_video,download_status,attempts_used,error_message,fps_out,frame_count_out,video_width_out,video_height_out,duration_sec_out,row_index
0,match [light-a-MATCH],match,0.000,0,0,0,83,match light-a-MATCH,830,360.0,30.000,2.767,https://www.youtube.com/watch?v=C37R_Ix8-qs,match,"[0.05754461884498596, 0.21637457609176636, 1.0...",640.0,NaN,train,https://www.youtube.com/watch?v=C37R_Ix8-qs,C37R_Ix8-qs,C37R_Ix8-qs,MS-ASL/videos_MS_ASL_raw/msasl_raw_C37R_Ix8-qs...,True,already_exists,0,,30.00000,85.0,640.0,360.0,2.833333,0
1,FAIL,fail,0.000,0,-1,0,74,FAIL,542,360.0,25.000,2.960,https://www.youtube.com/watch?v=PIsUJl8BN_I,fail,"[0.0657794177532196, 0.16717177629470825, 0.93...",480.0,NaN,train,https://www.youtube.com/watch?v=PIsUJl8BN_I,PIsUJl8BN_I,PIsUJl8BN_I,MS-ASL/videos_MS_ASL_raw/msasl_raw_PIsUJl8BN_I...,True,already_exists,0,,25.00000,76.0,480.0,360.0,3.040000,1
2,laugh,laugh,0.000,4,26,0,31,SignSchool Laugh with Legs 2,312,360.0,29.970,1.034,www.youtube.com/watch?v=9FdHlMOnVjg,laugh,"[0.13188594579696655, 0.32334136962890625, 1.0...",640.0,NaN,train,https://www.youtube.com/watch?v=9FdHlMOnVjg,9FdHlMOnVjg,9FdHlMOnVjg,,False,failed,1,"Command '['yt-dlp', '--no-playlist', '-f', 'be...",NaN,NaN,NaN,NaN,NaN,2
3,BOOK,book,0.000,0,-1,0,66,BOOK(3),38,360.0,25.000,2.640,https://www.youtube.com/watch?v=J7tP98oDxqE,book,"[0.05569887161254883, 0.25173279643058777, 0.9...",480.0,NaN,train,https://www.youtube.com/watch?v=J7tP98oDxqE,J7tP98oDxqE,J7tP98oDxqE,MS-ASL/videos_MS_ASL_raw/msasl_raw_J7tP98oDxqE...,True,already_exists,0,,25.00000,67.0,480.0,360.0,2.680000,3
4,sign-language,sign language,0.000,0,-1,0,75,SIGN-LANGUAGE-S-CLAW-F,848,360.0,29.970,2.502,www.youtube.com/watch?v=N2mG9ZKjrGA,sign language,"[0.03904399275779724, 0.24198183417320251, 1.0...",640.0,NaN,train,https://www.youtube.com/watch?v=N2mG9ZKjrGA,N2mG9ZKjrGA,N2mG9ZKjrGA,MS-ASL/videos_MS_ASL_raw/msasl_raw_N2mG9ZKjrGA...,True,already_exists,0,,29.97003,77.0,640.0,360.0,2.569233,4
5,school,school,1.101,1,44,33,110,ASL Vocabulary school,10,360.0,29.970,3.670,https://www.youtube.com/watch?v=1AyT77LqJzQ,school,"[0.0, 0.07651948928833008, 1.0, 1.0]",640.0,1.0,train,https://www.youtube.com/watch?v=1AyT77LqJzQ,1AyT77LqJzQ,1AyT77LqJzQ,,False,failed,1,"Command '['yt-dlp', '--no-playlist', '-f', 'be...",NaN,NaN,NaN,NaN,NaN,5
6,school,school,4.671,1,44,140,206,ASL Vocabulary school,10,360.0,29.970,6.874,https://www.youtube.com/watch?v=1AyT77LqJzQ,school,"[0.0, 0.07651948928833008, 1.0, 1.0]",640.0,1.0,train,https://www.youtube.com/watch?v=1AyT77LqJzQ,1AyT77LqJzQ,1AyT77LqJzQ,,False,failed,1,"Command '['yt-dlp', '--no-playlist', '-f', 'be...",NaN,NaN,NaN,NaN,NaN,6
7,easter,easter,0.000,2,58,0,116,Easter,794,360.0,29.595,3.920,https://www.youtube.com/watch?v=SVWABYmFdhs,easter,"[0.0, 0.12271708250045776, 1.0, 0.859626889228...",640.0,NaN,train,https://www.youtube.com/watch?v=SVWABYmFdhs,SVWABYmFdhs,SVWABYmFdhs,MS-ASL/videos_MS_ASL_raw/msasl_raw_SVWABYmFdhs...,True,already_exists,0,,29.59500,118.0,640.0,360.0,3.987160,7
8,Boring,boring,0.000,13,-1,0,71,ASL Boring,46,360.0,25.000,2.840,https://www.youtube.com/watch?v=CYx7qm62Zwo,bored,"[0.056927233934402466, 0.26684892177581787, 0....",640.0,NaN,train,https://www.youtube.com/watch?v=CYx7qm62Zwo,CYx7qm62Zwo,CYx7qm62Zwo,MS-ASL/videos_MS_ASL_raw/msasl_raw_CYx7qm62Zwo...,True,already_exists,0,,25.00000,72.0,640.0,360.0,2.880000,8
9,PAST,past,0.000,191,13,0,32,PAST,510,720.0,29.970,1.068,https://www.youtube.com/watch?v=cJOyCgIKyeA,past,"[0.06102180480957031, 0.2915322780609131, 0.99...",1280.0,NaN,train,https://www.youtube.com/watch?v=cJOyCgIKyeA,cJOyCgIKyeA,cJOyCgIKyeA,,False,failed,1,"Command '['yt-dlp', '--no-playlist', '-f', 'be...",NaN,NaN,NaN,NaN,NaN,9


In [5]:
target_columns = [
    'label',
    'source',
    'video_path',
    'start_frame',
    'end_frame',
    'length_frames',
    'duration_sec',
    'fps',
    'signer_id',
    'has_video',
    'video_width',
    'video_height',
]

out = manifest_df.copy()
out['label'] = out[LABEL_COLUMN].astype(str).str.strip().str.lower()
out['source'] = SOURCE_NAME
# local_video_path is already relative (returned by to_repo_path in process_video)
out['video_path'] = out['local_video_path'].astype(str)

out['start_frame'] = pd.to_numeric(out['start'], errors='coerce')
out['end_frame'] = pd.to_numeric(out['end'], errors='coerce')
length_from_range = out['end_frame'] - out['start_frame'] + 1
out['length_frames'] = np.where(length_from_range > 0, length_from_range, np.nan)

duration_from_time = pd.to_numeric(out['end_time'], errors='coerce') - pd.to_numeric(out['start_time'], errors='coerce')
duration_from_frames = np.where(
    out['length_frames'].notna() & pd.to_numeric(out['fps'], errors='coerce').notna() & (pd.to_numeric(out['fps'], errors='coerce') > 0),
    out['length_frames'] / pd.to_numeric(out['fps'], errors='coerce'),
    np.nan,
 )
out['duration_sec'] = np.where(duration_from_time > 0, duration_from_time, duration_from_frames)

out['fps'] = np.where(pd.to_numeric(out['fps'], errors='coerce').notna(), pd.to_numeric(out['fps'], errors='coerce'), out['fps_out'])
out['signer_id'] = pd.to_numeric(out['signer_id'], errors='coerce')
out['has_video'] = out['has_video'].fillna(False).astype(bool)
out['video_width'] = np.where(pd.to_numeric(out['width'], errors='coerce').notna(), pd.to_numeric(out['width'], errors='coerce'), out['video_width_out'])
out['video_height'] = np.where(pd.to_numeric(out['height'], errors='coerce').notna(), pd.to_numeric(out['height'], errors='coerce'), out['video_height_out'])

universal_df = out[target_columns].copy()
universal_df.to_csv(OUTPUT_UNIVERSAL_CSV, index=False)

universal_df_has_video = universal_df[universal_df['has_video'] == True].copy()
universal_df_has_video.to_csv(OUTPUT_UNIVERSAL_CSV_HAS_VIDEO, index=False)

print('Saved universal CSV (all rows):', OUTPUT_UNIVERSAL_CSV)
print('Rows in universal CSV (all rows):', len(universal_df))
print('Saved universal CSV (has_video only):', OUTPUT_UNIVERSAL_CSV_HAS_VIDEO)
print('Rows in universal CSV (has_video only):', len(universal_df_has_video))
display(universal_df.head(10))

Saved universal CSV (all rows): C:\Users\Magda\source\repos\private\szum\MS-ASL\msasl_universal_metadata_full_all_raw.csv
Rows in universal CSV (all rows): 25513
Saved universal CSV (has_video only): C:\Users\Magda\source\repos\private\szum\MS-ASL\msasl_universal_metadata_full_all_raw_has_video.csv
Rows in universal CSV (has_video only): 16816


,label,source,video_path,start_frame,end_frame,length_frames,duration_sec,fps,signer_id,has_video,video_width,video_height
0,match,msasl,MS-ASL/videos_MS_ASL_raw/msasl_raw_C37R_Ix8-qs...,0,83,84.0,2.767,30.000,0,True,640.0,360.0
1,fail,msasl,MS-ASL/videos_MS_ASL_raw/msasl_raw_PIsUJl8BN_I...,0,74,75.0,2.960,25.000,0,True,480.0,360.0
2,laugh,msasl,,0,31,32.0,1.034,29.970,4,False,640.0,360.0
3,book,msasl,MS-ASL/videos_MS_ASL_raw/msasl_raw_J7tP98oDxqE...,0,66,67.0,2.640,25.000,0,True,480.0,360.0
4,sign language,msasl,MS-ASL/videos_MS_ASL_raw/msasl_raw_N2mG9ZKjrGA...,0,75,76.0,2.502,29.970,0,True,640.0,360.0
5,school,msasl,,33,110,78.0,2.569,29.970,1,False,640.0,360.0
6,school,msasl,,140,206,67.0,2.203,29.970,1,False,640.0,360.0
7,easter,msasl,MS-ASL/videos_MS_ASL_raw/msasl_raw_SVWABYmFdhs...,0,116,117.0,3.920,29.595,2,True,640.0,360.0
8,boring,msasl,MS-ASL/videos_MS_ASL_raw/msasl_raw_CYx7qm62Zwo...,0,71,72.0,2.840,25.000,13,True,640.0,360.0
9,past,msasl,,0,32,33.0,1.068,29.970,191,False,1280.0,720.0


In [6]:
print('Row-level download status summary:')
display(manifest_df['download_status'].value_counts(dropna=False).to_frame('count'))

print('Unique-video status summary:')
display(manifest_df[['video_key', 'download_status']].drop_duplicates()['download_status'].value_counts(dropna=False).to_frame('count'))

print('has_video summary:')
display(universal_df['has_video'].value_counts(dropna=False).to_frame('count'))

failed = manifest_df[~manifest_df['download_status'].isin(['ok', 'already_exists'])].copy()
if len(failed) > 0:
    print('Failed sample:')
    display(failed[['row_index', 'split', 'clean_text', 'url_norm', 'download_status', 'attempts_used', 'error_message']].head(30))

Row-level download status summary:


,count
download_status,
already_exists,16816
failed,8697


Unique-video status summary:


,count
download_status,
already_exists,4102
failed,3110


has_video summary:


,count
has_video,
True,16816
False,8697


Failed sample:


,row_index,split,clean_text,url_norm,download_status,attempts_used,error_message
2,2,train,laugh,https://www.youtube.com/watch?v=9FdHlMOnVjg,failed,1,"Command '['yt-dlp', '--no-playlist', '-f', 'be..."
5,5,train,school,https://www.youtube.com/watch?v=1AyT77LqJzQ,failed,1,"Command '['yt-dlp', '--no-playlist', '-f', 'be..."
6,6,train,school,https://www.youtube.com/watch?v=1AyT77LqJzQ,failed,1,"Command '['yt-dlp', '--no-playlist', '-f', 'be..."
9,9,train,past,https://www.youtube.com/watch?v=cJOyCgIKyeA,failed,1,"Command '['yt-dlp', '--no-playlist', '-f', 'be..."
14,14,train,germany,https://www.youtube.com/watch?v=zqfcp6NRKpA,failed,1,"Command '['yt-dlp', '--no-playlist', '-f', 'be..."
15,15,train,like,https://www.youtube.com/watch?v=7y5Ye-2-ZBs,failed,1,"Command '['yt-dlp', '--no-playlist', '-f', 'be..."
29,29,train,portugal,https://www.youtube.com/watch?v=rYW7VFPi9_w,failed,1,"Command '['yt-dlp', '--no-playlist', '-f', 'be..."
54,54,train,yellow,https://www.youtube.com/watch?v=0Beq_NIDj2c,failed,1,"Command '['yt-dlp', '--no-playlist', '-f', 'be..."
55,55,train,yellow,https://www.youtube.com/watch?v=0Beq_NIDj2c,failed,1,"Command '['yt-dlp', '--no-playlist', '-f', 'be..."
64,64,train,again,https://www.youtube.com/watch?v=AoQAPgEUIAs,failed,1,"Command '['yt-dlp', '--no-playlist', '-f', 'be..."


## Next step

Use msasl_universal_metadata_full_all.csv for merge prep and filtering in the next notebook stage.